In [10]:
import time
from pathlib import Path

import numpy as np
import pandas as pd
from pymatgen.core import Composition, Element

# ---------- settings ----------
INPUT_FILE = "Toxicity.csv"         
REFERENCE_FILE = "RawData.csv"      
EXPT_FILE = "expt_gap.csv"          
OUTPUT_FILE = "Novel.csv"           

KNOWN_LEVEL = "formula"             
                                    
REMOVE_KNOWN_EXPT = True            
REMOVE_KNOWN_ICSD = True            
REMOVE_NOT_NEUTRAL = True           
TRY_EXTENDED_OXI = True             
MAX_ATOMS_EXACT = 80                
# ------------------------------

HERE = Path.cwd()
candidates = [HERE / "data", HERE, HERE / "notebook" / "data" / "data", HERE / "notebook" / "data"]
DATA_DIR = next((p for p in candidates if (p / INPUT_FILE).exists()), None)
assert DATA_DIR is not None, f"{INPUT_FILE} not found. Run the earlier notebooks first."
IN_PATH = DATA_DIR / INPUT_FILE
OUT_PATH = DATA_DIR / OUTPUT_FILE

def reduced(f):
    try:
        c = Composition(str(f))
        if not all(isinstance(e, Element) for e in c.elements):
            return None
        return c.reduced_formula
    except Exception:
        return None

df = pd.read_csv(IN_PATH)
assert "formula" in df.columns and "theoretical" in df.columns, "Need formula and theoretical columns"
df["reduced_formula"] = df["formula"].map(reduced)
print("Read", len(df), "materials from", IN_PATH.name, "|", df["reduced_formula"].nunique(), "different formulas")

Read 2111 materials from Toxicity.csv | 1451 different formulas


In [11]:
ref_path = DATA_DIR / REFERENCE_FILE
if ref_path.exists():
    ref = pd.read_csv(ref_path)
    print("Reference for known formulas:", REFERENCE_FILE, f"({len(ref):,} rows)")
else:
    ref = df.copy()
    print(f"WARNING: {REFERENCE_FILE} not found, using {INPUT_FILE} only (may miss some known formulas)")
ref["reduced_formula"] = ref["formula"].map(reduced)
observed = ref["theoretical"].astype(str).str.lower().eq("false")        # False = seen experimentally (ICSD)
known_icsd_formulas = set(ref.loc[observed, "reduced_formula"].dropna())

# --- B. formulas in the experimental dataset ---
expt = pd.read_csv(DATA_DIR / EXPT_FILE)
fcol = next(c for c in expt.columns if "formula" in c.lower())
known_expt_formulas = set(expt[fcol].map(reduced).dropna())

print(f"Formulas with an experimentally reported structure (MP): {len(known_icsd_formulas):,}")
print(f"Formulas in the experimental gap data                  : {len(known_expt_formulas):,}")

Reference for known formulas: RawData.csv (76,824 rows)
Formulas with an experimentally reported structure (MP): 21,854
Formulas in the experimental gap data                  : 4,920


In [12]:
if KNOWN_LEVEL == "formula":
    df["known_icsd"] = df["reduced_formula"].isin(known_icsd_formulas)
else:
    df["known_icsd"] = df["theoretical"].astype(str).str.lower().eq("false")
df["known_expt"] = df["reduced_formula"].isin(known_expt_formulas)

known = pd.Series(False, index=df.index)
if REMOVE_KNOWN_ICSD:
    known |= df["known_icsd"]
if REMOVE_KNOWN_EXPT:
    known |= df["known_expt"]
df["known_reason"] = np.select(
    [df["known_icsd"] & df["known_expt"], df["known_icsd"], df["known_expt"]],
    ["MP experimental structure + expt gap data", "MP experimental structure (ICSD)", "in expt gap data"], default="")

known_df = df[known].reset_index(drop=True)
new_df = df[~known].reset_index(drop=True)

print(f"Input rows                      : {len(df):,}  ({df['reduced_formula'].nunique():,} formulas)")
print(f"Removed, already synthesised    : {len(known_df):,}  ({known_df['reduced_formula'].nunique():,} formulas)")
print(known_df["known_reason"].value_counts().to_string())
print(f"Left (never reported)           : {len(new_df):,}  ({new_df['reduced_formula'].nunique():,} formulas)")

Input rows                      : 2,111  (1,451 formulas)
Removed, already synthesised    : 1,249  (912 formulas)
known_reason
MP experimental structure (ICSD)             1056
MP experimental structure + expt gap data     172
in expt gap data                               21
Left (never reported)           : 862  (539 formulas)


In [13]:
new_df = new_df.sort_values(["reduced_formula", "energy_above_hull", "formation_energy_per_atom"]).reset_index(drop=True)
grp = new_df.groupby("reduced_formula", sort=False)
extra = pd.DataFrame({
    "n_structures": grp.size(),
    "space_groups": grp["space_group"].agg(lambda s: ";".join(sorted(set(map(str, s))))) if "space_group" in new_df.columns else "",
    "ehull_min": grp["energy_above_hull"].min(),
    "ehull_max": grp["energy_above_hull"].max(),
    "gap_pbe_min": grp["band_gap_pbe"].min() if "band_gap_pbe" in new_df.columns else np.nan,
    "gap_pbe_max": grp["band_gap_pbe"].max() if "band_gap_pbe" in new_df.columns else np.nan,
}).reset_index()

merged = new_df.drop_duplicates("reduced_formula", keep="first").merge(extra, on="reduced_formula", how="left")
merged = merged.drop(columns=["known_icsd", "known_expt", "known_reason"]).reset_index(drop=True)

print(f"Rows before merging : {len(new_df):,}")
print(f"Rows after merging  : {len(merged):,}  (one row per formula)")
print("Formulas with more than one structure:", int((merged["n_structures"] > 1).sum()))
merged[["formula", "band_gap", "energy_above_hull", "n_structures", "space_groups"]].head(10)

Rows before merging : 862
Rows after merging  : 539  (one row per formula)
Formulas with more than one structure: 114


,formula,band_gap,energy_above_hull,n_structures,space_groups
0,Al(VS2)4,1.663,0.000000,1,Imm2
1,Al2FeO4,1.938,0.014607,3,C2/m;Cm;P3m1
2,Al3BC3,2.943,0.000000,2,P6_3/mmc;P6_3cm
3,Al4SiC4,2.557,0.047235,1,P6_3mc
4,Al5CuSe8,2.381,0.029427,1,F-43m
5,AlAuO2,2.569,0.012329,1,P6_3/mmc
6,AlIn2N3,2.677,0.015751,1,Cmc2_1
7,AlInSe3,2.398,0.000000,1,P6_1
8,BSe2,2.991,0.000000,1,P2_1/c
9,Ba(AgO)2,1.991,0.001692,1,I4_1/amd


In [14]:
def fast_balance(comp, all_states):
    """Fast check: can the total charge be zero, using the lowest and highest oxidation state of each element?"""
    low = high = 0.0
    for el, n in comp.get_el_amt_dict().items():
        e = Element(el)
        states = e.oxidation_states if all_states else e.common_oxidation_states
        if not states:
            return False
        low += n * min(states)
        high += n * max(states)
    return low <= 1e-9 and high >= -1e-9

def balance(formula):
    """Return (balanced, oxidation states, method). Common oxidation states first, then all of them."""
    try:
        comp = Composition(formula)
    except Exception:
        return False, "", ""
    big = comp.reduced_composition.num_atoms > MAX_ATOMS_EXACT
    for method, allox in (("common", False), ("extended", True)):
        if method == "extended" and not TRY_EXTENDED_OXI:
            break
        if big:
            if fast_balance(comp, allox):
                return True, "large formula (fast check)", method
            continue
        try:
            guesses = comp.oxi_state_guesses(all_oxi_states=allox, max_sites=-1)
        except Exception:
            guesses = []
        guesses = [g for g in guesses if any(abs(v) > 1e-9 for v in g.values())]   # all-zero = not a real ionic balance
        if guesses:
            best = guesses[0]
            return True, ";".join(f"{el}{round(v, 2):+g}" for el, v in best.items()), method
    return False, "", ""

t0 = time.time()
res = merged["reduced_formula"].map(balance)
merged["charge_neutral"] = res.map(lambda r: r[0])
merged["oxi_states"] = res.map(lambda r: r[1])
merged["oxi_method"] = res.map(lambda r: r[2])

if REMOVE_NOT_NEUTRAL:
    not_neutral_df = merged[~merged["charge_neutral"]].reset_index(drop=True)
    final = merged[merged["charge_neutral"]].reset_index(drop=True)
else:
    not_neutral_df = merged.iloc[0:0]
    final = merged

print(f"Checked {len(merged):,} formulas in {time.time() - t0:.0f} s")
print(f"Charge neutral (common oxidation states) : {int((merged['oxi_method'] == 'common').sum()):,}")
print(f"Charge neutral (only with unusual states): {int((merged['oxi_method'] == 'extended').sum()):,}")
print(f"Not charge neutral (removed)             : {len(not_neutral_df):,}")
print(f"Left                                     : {len(final):,}")
final[["formula", "band_gap", "oxi_states", "oxi_method"]].head(10)

Checked 539 formulas in 587 s
Charge neutral (common oxidation states) : 506
Charge neutral (only with unusual states): 31
Not charge neutral (removed)             : 2
Left                                     : 537


,formula,band_gap,oxi_states,oxi_method
0,Al(VS2)4,1.663,Al+3;V+3.25;S-2,common
1,Al2FeO4,1.938,Al+3;Fe+2;O-2,common
2,Al3BC3,2.943,Al+3;B+3;C-4,common
3,Al4SiC4,2.557,Al+3;Si+4;C-4,common
4,Al5CuSe8,2.381,Al+3;Cu+1;Se-2,common
5,AlAuO2,2.569,Al+3;Au+1;O-2,extended
6,AlIn2N3,2.677,Al+3;In+3;N-3,common
7,AlInSe3,2.398,Al+3;In+3;Se-2,common
8,BSe2,2.991,B+3;Se-1.5,common
9,Ba(AgO)2,1.991,Ba+2;Ag+1;O-2,common


In [15]:
final.to_csv(OUT_PATH, index=False)
print("Saved:", OUT_PATH, "| exists:", OUT_PATH.exists(), "| rows:", len(final))
print()
print(f"Input rows                   : {len(df):,}")
print(f"Removed (already synthesised): {len(known_df):,}")
print(f"After merging repeats        : {len(merged):,}")
print(f"Removed (not charge neutral) : {len(not_neutral_df):,}")
print(f"Final (Novel.csv)            : {len(final):,}")

Saved: c:\Users\solom\Desktop\SemiconductorProject\notebook\data\data\Novel.csv | exists: True | rows: 537

Input rows                   : 2,111
Removed (already synthesised): 1,249
After merging repeats        : 539
Removed (not charge neutral) : 2
Final (Novel.csv)            : 537
